# 07 — Order Recommendations
**RetailMind · Data Science Practicum 2 · Sai Teja Sunku**

Turns forecasts into actionable inventory decisions using the
**periodic-review (R, S) inventory model**:

```
mean_demand = mean of forecast over (lead_time + review_period) days
std_demand = std of forecast over the same window
target_stock_S = mean × (L + R) + z(SL) × std × √(L + R)
reorder_qty = max(0, target_S − on_hand)
```

| Output | Meaning |
|---|---|
| `urgency` | urgent / reorder soon / stocked |
| `days_of_cover` | At current daily demand, how long on-hand stock lasts |
| `reorder_point` | Continuous-review trigger (informational) |
| `target_stock_level` | Order *up to* this level |
| `recommended_order_qty` | What to order now |


In [1]:
# Common setup: make the project package importable from the notebooks/ folder
import sys, warnings, json
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px


## 1. Generate forecast, then convert to reorder decisions

In [2]:
from retailmind import RetailPipeline, RecommendationParams
from retailmind.recommend import recommend_orders, summary_metrics

p = RetailPipeline.from_files('../train.csv', auxiliary_paths=['../store.csv'])
p.canonicalize_()
p.horizon = 14
p.max_entities_for_full_forecast = 5
p.forecast_(sample_entities=30, cv_folds=2)

params = RecommendationParams(lead_time_days=7, service_level=0.95, review_period_days=7)
recs = recommend_orders(p.forecast, params=params)
recs

,entity_id,urgency,on_hand,days_of_cover,mean_daily_demand,demand_std,lead_time,review_period,service_level,reorder_point,target_stock_level,recommended_order_qty
0,2,🔴 urgent,0.0,0.0,24325.1,1575.3,7,7,0.95,177131.0,350246.2,350246.2
1,0,🔴 urgent,0.0,0.0,23424.0,3026.9,7,7,0.95,177140.9,346565.4,346565.4
2,4,🔴 urgent,0.0,0.0,19564.7,2998.4,7,7,0.95,150001.7,292359.6,292359.6
3,3,🔴 urgent,0.0,0.0,17991.2,1895.0,7,7,0.95,134185.4,263540.0,263540.0
4,1,🔴 urgent,0.0,0.0,17690.5,2117.4,7,7,0.95,133048.4,260698.8,260698.8


## 2. Headline summary

In [3]:
sm = summary_metrics(recs)
print(json.dumps(sm, indent=2))

{
  "total_entities": 5,
  "n_urgent": 5,
  "n_reorder_soon": 0,
  "n_stocked": 0,
  "total_order_qty": 1513410.0000000002,
  "total_target_stock": 1513410.0000000002,
  "total_on_hand": 0.0
}


## 3. Effect of changing service level
Higher service level → larger safety stock → bigger order quantities.

In [4]:
rows = []
for sl in [0.80, 0.90, 0.95, 0.99]:
    rec = recommend_orders(p.forecast, params=RecommendationParams(
        lead_time_days=7, service_level=sl, review_period_days=7))
    rows.append({'service_level': sl, 'total_order_qty': rec['recommended_order_qty'].sum()})
pd.DataFrame(rows)

,service_level,total_order_qty
0,0.80,1478507.8
1,0.90,1497623.7
2,0.95,1513410.0
3,0.99,1543022.3


## 4. Simulate having some on-hand stock

In [5]:
ents = p.forecast['entity_id'].unique()
oh = pd.DataFrame({'entity_id': [str(e) for e in ents],
                    'on_hand': np.random.RandomState(42).uniform(5000, 50000, len(ents))})
with_stock = recommend_orders(p.forecast, on_hand=oh, params=params)
with_stock[['entity_id', 'urgency', 'on_hand', 'days_of_cover', 'recommended_order_qty']]

,entity_id,urgency,on_hand,days_of_cover,recommended_order_qty
0,0,🔴 urgent,21854.3,0.9,324711.1
1,2,🔴 urgent,37939.7,1.6,312306.5
2,4,🔴 urgent,12020.8,0.6,280338.8
3,3,🔴 urgent,31939.6,1.8,231600.3
4,1,🔴 urgent,47782.1,2.7,212916.6


## Summary
- Inventory model is mathematically consistent (single (R, S) framework)
- User can input current stock → recommendations adapt
- Urgency classification turns numbers into actions

**Next:** [08 — Assistant](08_assistant.ipynb)
